# Stage 5: building the three networks

**What this stage does:** groups the partnerships into a node table and an edge table for each format, and
checks that the networks match the partnership table exactly.

**Stop rule:** if any of these checks fails, the notebook stops.

**Reads:** `data/processed/partnership_observations.csv`
**Writes:** `data/processed/nodes_<format>.csv`, `edges_<format>.csv`, `players.csv` and
`outputs/tables/validation_aggregation_checks.csv`

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_04_network_building.ipynb
stage_log = get_stage_logger("05_networks")

In [2]:
observations = read_table(PROCESSED_FOLDER / "partnership_observations.csv")
player_table = build_player_table(observations)
node_table = build_node_table(observations, player_table)
edge_table = build_edge_table(observations, player_table)
write_table(player_table, PROCESSED_FOLDER / "players.csv")

### One node table and one edge table per format, each checked against the partnerships

In [3]:
aggregation_results = []
for format_name in FORMATS:
    file_tag = FORMAT_FILE_TAG[format_name]
    write_table(node_table[node_table["format"] == format_name], PROCESSED_FOLDER / f"nodes_{file_tag}.csv")
    write_table(edge_table[edge_table["format"] == format_name], PROCESSED_FOLDER / f"edges_{file_tag}.csv")
    for check_name, passed, detail in aggregation_checks(observations, node_table, edge_table, format_name):
        aggregation_results.append({"format": format_name, "check": check_name, "passed": passed, "detail": detail})
    partnership_graph = build_format_graph(node_table, edge_table, format_name)
    stage_log.info("%s: %d nodes, %d edges, total weight %d", format_name, partnership_graph.number_of_nodes(),
                   partnership_graph.number_of_edges(), int(partnership_graph.size(weight="weight")))

aggregation_results = pd.DataFrame(aggregation_results)
write_table(aggregation_results, TABLES_FOLDER / "validation_aggregation_checks.csv")

2026-11-19 18:05:10,316 INFO partnerships.05_networks: Test: 51 nodes, 335 edges, total weight 43080


2026-11-19 18:05:10,331 INFO partnerships.05_networks: ODI: 50 nodes, 262 edges, total weight 22929


2026-11-19 18:05:10,344 INFO partnerships.05_networks: T20I: 44 nodes, 222 edges, total weight 18974


### Stop rule

In [4]:
# If the same player appeared under two spellings, we would see it here (the IDs merge them anyway).
players_with_several_spellings = player_table[player_table["name_variants"].str.contains("|", regex=False)]
stage_log.info("players with more than one spelling of their name: %d", len(players_with_several_spellings))
all_checks_passed = bool(aggregation_results["passed"].all())
stage_log.info("network checks passed: %d/%d, CHECKS %s", aggregation_results["passed"].sum(),
               len(aggregation_results), "PASSED" if all_checks_passed else "FAILED")
assert all_checks_passed, "The networks do not match the partnership table"

2026-11-19 18:05:10,348 INFO partnerships.05_networks: players with more than one spelling of their name: 0


2026-11-19 18:05:10,349 INFO partnerships.05_networks: network checks passed: 36/36, CHECKS PASSED
